# Stage 1 - terms correction

Theree are 3 wariants to be compared:

1. Whisper
2. Whisper + hotwords
3. Whisper + hotwords + term dictionary with rapidfuzz
4. Whipser + hotwords + term dict + SLM

Each run with its artifacts is being logged help compare different approaches.


## Wariant 1

This is the baseline for the stage. It will measure the raw performance without any enhancements.


In [ ]:
from faster_whisper import WhisperModel
import librosa
import numpy as np
import pandas as pd

from constants import DICTIONARY_PATH, AUDIO_DIR, RECORDINGS_PATH
from log_runs import WhisperConfig, DictionaryConfig, SlmConfig, RunConfig

recordings = pd.read_csv(RECORDINGS_PATH)

slmm_config: SlmConfig | None = None
dict_config: DictionaryConfig | None = None
whisper_config: WhisperConfig = WhisperConfig(
    model="small",
    device="cpu",
    compute_type="int8",
    language="pl",
    beam_size=5,
    vad_filter=False,
    word_timestamps=True,
    temperature=1
)


model = WhisperModel("large-v3-turbo",
                     device="cpu", compute_type="int8")

verbatim_prompt = "Yyy, no więc, eee, myślę że... hmm, to znaczy, yyy, tak jakby, no wiesz, uhm, dokładnie tak."

segments, info = model.transcribe(
    "../data/audio/rec_001.wav",
    language="pl",
    word_timestamps=True,
    vad_filter=False,
    hotwords="RAG,Vercel,python,POC,Docker",

    beam_size=3,
    no_speech_threshold=0.9,
    temperature=0,
    log_prob_threshold=None,
    compression_ratio_threshold=None,
    initial_prompt=verbatim_prompt
)

words = [w for seg in segments for w in seg.words]

/opt/miniconda3/envs/interview-sim/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
y, sr = librosa.load("../data/audio/rec_001.wav", sr=16000)
hop = 160  # 10 ms
rms = librosa.feature.rms(y=y, frame_length=400, hop_length=hop)[0]
t = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)

noise = np.percentile(rms, 10)  # przybliżony poziom ciszy w tym nagraniu

for a, b in zip(words, words[1:]):
    gap = b.start - a.end
    if gap > 0.25:
        m = (t >= a.end) & (t <= b.start)
        ratio = rms[m].mean() / noise
        print(f"{a.end:.2f}-{b.start:.2f}  luka {gap:.2f}s  RMS/cisza = {ratio:.1f}")

MIN_GAP = 0.20   # s – krótsze luki ignorujemy (naturalna artykulacja)
FILLER_RATIO = 2.2  # RMS/cisza powyżej tego = "yyy". Wartość startowa, do dostrojenia!

out = []
for i, w in enumerate(words):
    out.append(w.word.strip())
    if i + 1 == len(words):
        break
    nxt = words[i + 1]
    gap = nxt.start - w.end
    if gap < MIN_GAP:
        continue
    m = (t >= w.end) & (t <= nxt.start)
    ratio = rms[m].mean() / noise
    if ratio >= FILLER_RATIO:
        out.append(f"[yyy {gap:.1f}s]")   # głos w luce
    else:
        out.append(f"[cisza {gap:.1f}s]")  # pauza cicha

print(" ".join(out))

4.52-5.08  luka 0.56s  RMS/cisza = 74.3
9.08-9.84  luka 0.76s  RMS/cisza = 11.8
12.44-13.44  luka 1.00s  RMS/cisza = 33.6
16.20-16.60  luka 0.40s  RMS/cisza = 31.8
19.68-20.08  luka 0.40s  RMS/cisza = 76.9
21.38-22.02  luka 0.64s  RMS/cisza = 25.3
25.00-25.28  luka 0.28s  RMS/cisza = 17.6
30.26-31.42  luka 1.16s  RMS/cisza = 54.2
35.68-36.44  luka 0.76s  RMS/cisza = 30.6
38.24-38.92  luka 0.68s  RMS/cisza = 32.1
42.10-42.80  luka 0.70s  RMS/cisza = 41.8
43.80-44.26  luka 0.46s  RMS/cisza = 139.5
48.96-49.86  luka 0.90s  RMS/cisza = 26.4
52.32-53.38  luka 1.06s  RMS/cisza = 39.7
59.76-60.14  luka 0.38s  RMS/cisza = 97.1
62.78-65.38  luka 2.60s  RMS/cisza = 20.1
68.68-69.16  luka 0.48s  RMS/cisza = 0.4
Jeśli chodzi o projekt, z którego jestem najbardziej dumny, to jest to platforma Vokare, [yyy 0.6s] która jest wdrożona produkcyjnie i można z niej korzystać. [yyy 0.8s] Mamy już nawet stałych użytkowników [yyy 1.0s] i codziennie rejestrujemy ruch na platformie. [yyy 0.4s] Jest to aplikacj